In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import time
import re
import json
import unicodedata
from datetime import datetime

# -----------------------
# Config
# -----------------------
BASE_URL = "https://www.motherjones.com/politics/"
PAGE_URL_TEMPLATE = "https://www.motherjones.com/politics/page/{}/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; MJ-Scraper/1.0)"
}

REQUEST_DELAY = 1.5
MAX_LINKS = 200

# -----------------------
# Utilities
# -----------------------
def utc_now_iso():
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def compute_article_id(url):
    slug = urlparse(url).path.rstrip("/").split("/")[-1]
    return slug.replace("-", "_")

def dump_articles_to_json(articles, output_path):
    """
    Dump extracted articles to a JSON file.

    Parameters:
        articles (list): List of article dicts
        output_path (str): Path to output JSON file
    """
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            articles,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"[output] Saved {len(articles)} articles to {output_path}")

# -----------------------
# Pagination: extract links
# -----------------------
def extract_article_links(html):
    soup = BeautifulSoup(html, "html.parser")
    links = set()

    for ul in soup.find_all("ul", class_="articles-list"):
        for a in ul.find_all("a", href=True):
            href = a["href"]

            if (
                href.startswith("https://www.motherjones.com/")
                and "/politics/" in href
                and re.search(r"/20\d{2}/\d{2}/", href)
            ):
                links.add(href.split("?")[0])

    return list(links)

def crawl_all_pages(max_links=100):
    page = 1
    all_links = set()

    while True:
        url = BASE_URL if page == 1 else PAGE_URL_TEMPLATE.format(page)
        print(f"[pagination] Fetching page {page}")

        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            break

        links = extract_article_links(r.text)
        if not links:
            break

        for link in links:
            all_links.add(link)
            if len(all_links) >= max_links:
                print(f"Reached {max_links} unique links.")
                return list(all_links)

        print(f"  Total unique links so far: {len(all_links)}")
        page += 1
        time.sleep(REQUEST_DELAY)

    return sorted(all_links)

# -----------------------
# Common body cleaning
# -----------------------

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)
    
def normalize_body_text(text):
    text = normalize_unicode(text)

    # Remove bracketed ellipses or descriptive tags
    text = re.sub(r"\[\s*\.\.\.\s*\]", " ", text)
    text = re.sub(r"\[([^\]]+)\]", r"\1", text)
    text = re.sub(r"\((cough|laughter|applause|pause)\)", " ", text, flags=re.I)

    # Normalize smart quotes
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )

    text = text.replace("–", "—")  # only unify dash type, no spacing

    # Keep invisible char cleanup
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
    text = text.replace("\u00A0", " ")

    # Fix cases like there'is -> there's
    text = re.sub(r"(\w)'(\w)", r"\1'\2", text)

    return clean_text(text)

def should_skip_paragraph(txt):
    txt_lower = txt.lower()

    return (
        txt_lower.startswith("top image:")
        or "mother jones illustration" in txt_lower
        or "zuma" in txt_lower
        or "getty" in txt_lower
        or "cover images" in txt_lower
        or "planet pix" in txt_lower
        or "wikimedia" in txt_lower
        or txt_lower.startswith("this story was originally published")
        or txt_lower.startswith("correction:")
        or txt_lower.startswith("editor's note:")
    )

def extract_published_date(soup):
    flag = soup.find("div", class_="mj-top-flag")
    if not flag:
        return None

    date_span = flag.find("span", class_="dateline")
    if not date_span:
        return None

    return clean_text(date_span.get_text())

# -----------------------
# Template A: fullwidth-body
# -----------------------
def extract_article_fullwidth(soup, url):
    h1 = soup.find("h1", class_="entry-title")
    body_div = soup.find("div", id="fullwidth-body")

    crawl_timestamp = utc_now_iso()

    if not h1 or not body_div:
        return None

    published_date = extract_published_date(soup)
    
    header = normalize_body_text(clean_text(h1.get_text()))

    for figcaption in body_div.find_all("figcaption"):
        figcaption.decompose()

    paragraphs = []
    for p in body_div.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt or should_skip_paragraph(txt):
            continue
        paragraphs.append(txt)

    body_text = normalize_body_text("\n\n".join(paragraphs))

    return {
        "article_id": compute_article_id(url),
        "header": header,
        "body_text": body_text,
        "word_count": len(body_text.split()),
        "outlet": "Mother Jones",
        "label": "left",
        "url": url,
        "published_date": published_date,
        "crawl_timestamp": crawl_timestamp,
    }

# -----------------------
# Template B: entry-content
# -----------------------
def extract_article_entry_content(soup, url):
    h1 = soup.find("h1", class_="entry-title")
    article = soup.find("article", class_="entry-content")

    crawl_timestamp = utc_now_iso()

    if not h1 or not article:
        return None

    published_date = extract_published_date(soup)
    
    header = normalize_body_text(clean_text(h1.get_text()))

    hero = article.find("div", class_="hero")
    if hero:
        hero.decompose()

    for figcaption in article.find_all("figcaption"):
        figcaption.decompose()

    for div in article.find_all(
        ["div", "aside", "section"],
        class_=re.compile(
            r"(ad|promo|widget|sidebar|freestar|membership)",
            re.I,
        ),
    ):
        div.decompose()

    paragraphs = []
    for p in article.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt or should_skip_paragraph(txt):
            continue
        paragraphs.append(txt)

    body_text = normalize_body_text("\n\n".join(paragraphs))

    return {
        "article_id": compute_article_id(url),
        "header": header,
        "body_text": body_text,
        "word_count": len(body_text.split()),
        "outlet": "Mother Jones",
        "label": "left",
        "url": url,
        "published_date": published_date,
        "crawl_timestamp": crawl_timestamp,
    }

# -----------------------
# Unified extractor
# -----------------------
def extract_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    article = extract_article_fullwidth(soup, url)
    if article:
        return article

    return extract_article_entry_content(soup, url)

# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    links = crawl_all_pages(max_links=MAX_LINKS)
    print(f"\nCollected {len(links)} article links\n")

    articles = []
    for i, link in enumerate(links, 1):
        print(f"[{i}/{len(links)}] Scraping")
        article = extract_article(link)
        if article:
            articles.append(article)
        time.sleep(REQUEST_DELAY)

    print(f"\nExtracted {len(articles)} articles\n")

    if articles:
        dump_articles_to_json(
            articles,
            output_path="mother_jones_articles.json"
        )

[pagination] Fetching page 1
  Total unique links so far: 20
[pagination] Fetching page 2
  Total unique links so far: 40
[pagination] Fetching page 3
  Total unique links so far: 60
[pagination] Fetching page 4
  Total unique links so far: 80
[pagination] Fetching page 5
  Total unique links so far: 100
[pagination] Fetching page 6
  Total unique links so far: 120
[pagination] Fetching page 7
  Total unique links so far: 140
[pagination] Fetching page 8
  Total unique links so far: 160
[pagination] Fetching page 9
  Total unique links so far: 180
[pagination] Fetching page 10
Reached 200 unique links.

Collected 200 article links

[1/200] Scraping
[2/200] Scraping
[3/200] Scraping
[4/200] Scraping
[5/200] Scraping
[6/200] Scraping
[7/200] Scraping
[8/200] Scraping
[9/200] Scraping
[10/200] Scraping
[11/200] Scraping
[12/200] Scraping
[13/200] Scraping
[14/200] Scraping
[15/200] Scraping
[16/200] Scraping
[17/200] Scraping
[18/200] Scraping
[19/200] Scraping
[20/200] Scraping
[21/200] 